# Lab 7: Model Selection, Inference, and Transformations

In `Hitters`, you'll study the relationship between major-league experience and salary. The lab follows two decisions that affect regression results: which predictors to keep and whether to log the response. You'll use simulation to see how those decisions affect false positives and power.


In [ ]:
options(repr.plot.width=10, repr.plot.height=5)
source("lab7_packages.R")
if (lab7_ensure_packages()) library(ottr)
hitters <- read.csv("hitters.csv", stringsAsFactors=TRUE)
hitters$League <- factor(hitters$League)
hitters$Division <- factor(hitters$Division)
hitters$NewLeague <- factor(hitters$NewLeague)
head(hitters)


Of the original 322 players, 263 have salary data. Those are the rows in `hitters`. We'll use these columns:

| Column | Description |
|--------|-------------|
| `Salary` | 1987 salary, in thousands of dollars |
| `Years` | Years in the major leagues |
| `Hits` | Hits in 1986 |
| `HmRun` | Home runs in 1986 |
| `CHmRun` | Career home runs |
| `Runs` | Runs in 1986 |
| `RBI` | Runs batted in in 1986 |
| `Walks` | Walks in 1986 |
| `CAtBat` | Career at-bats |
| `PutOuts` | Putouts in 1986 |
| `League` | League at the end of 1986 (A or N) |
| `Division` | Division at the end of 1986 (E or W) |

`Salary` is the response, and `Years` is our primary predictor.


## Part 1: Model Selection and False Positives

Lecture 13 showed that choosing a model with p-values can affect a test reported from that model. We'll drop a secondary predictor when its p-value is large, then report the primary predictor's p-value from the model we keep. When predictors are correlated, the errors in their coefficient estimates are related, so selection can distort the reported test.

The predictors are fixed `Hitters` columns. We'll generate independent Normal responses unrelated to them, so every rejection is a false positive. The three pairs below cover weak, moderate, and strong correlation.


In [ ]:
correlations_q13 <- c("Years/Hits"=cor(hitters$Years,hitters$Hits),
                      "Years/CHmRun"=cor(hitters$Years,hitters$CHmRun),
                      "Years/CAtBat"=cor(hitters$Years,hitters$CAtBat))
round(correlations_q13,3)


**Question 1.1.** The team plans to fit a model with two predictors, drop the second if its p-value exceeds 0.05, and report the resulting `Years` p-value. Which pair would you expect to give the greatest Type I error inflation in this null simulation?

1. `Years/Hits`, because weakly correlated predictors produce the largest distortion after selection.
2. `Years/CHmRun`, because moderate correlation is always more problematic than weak or strong correlation.
3. `Years/CAtBat`, because the coefficient estimates are most strongly related, so selection has the greatest effect on the reported `Years` test.
4. All three keep a 5% Type I error rate whenever the second predictor has no true effect.

Set `q1_1_ans` to one number.


_Points:_ 0.5

In [ ]:
q1_1_ans <- NULL # YOUR CODE HERE


In [ ]:
. = ottr::check("tests/q1_1.R")

**Question 1.2.** Write `sim_select_then_infer(reps, x1, x2)`. Each replicate uses a fresh, independent standard Normal response of length `length(x1)`, with the predictors held fixed. Record the `x1` p-value from the full model and from the selected model. Drop `x2` only when its full-model p-value exceeds 0.05; both models include an intercept.

Return a list with `pval_full` and `pval_selected`: vectors of `reps` unrounded, two-sided p-values, paired by replicate. Assume numeric covariates of equal length, at least four observations, no missing values, and no perfect collinearity.


_Points:_ 3

In [ ]:
sim_select_then_infer <- function(reps, x1, x2) {
    # YOUR CODE HERE
}


In [ ]:
set.seed(414)
check_q12 <- sim_select_then_infer(200,hitters$Years,hitters$CHmRun)
c(full=mean(check_q12$pval_full < .05),selected=mean(check_q12$pval_selected < .05))


In [ ]:
. = ottr::check("tests/q1_2.R")

**Question 1.3.** Write `selection_rates(results, correlations, alpha=0.05)`. `results` is a named list of outputs from your simulation; `correlations` is a numeric vector with matching names.

Return a data frame with columns `pair`, `cor_x1_x2`, `rate_full`, `rate_selected`, and `inflation`. The two rates are fractions of p-values below `alpha`; `inflation` is `rate_selected / alpha`. Sort by increasing absolute correlation. The number of pairs and the value of `alpha` may change.

The cell below runs the three comparisons and displays your table.


_Points:_ 1.5

In [ ]:
selection_rates <- function(results, correlations, alpha=0.05) {
    # YOUR CODE HERE
}


In [ ]:
set.seed(414)
sim_low <- sim_select_then_infer(2000,hitters$Years,hitters$Hits)
sim_med <- sim_select_then_infer(2000,hitters$Years,hitters$CHmRun)
sim_high <- sim_select_then_infer(2000,hitters$Years,hitters$CAtBat)
rates_q13 <- selection_rates(list("Years/Hits"=sim_low,"Years/CHmRun"=sim_med,
                                "Years/CAtBat"=sim_high),correlations_q13)
rates_q13


In [ ]:
. = ottr::check("tests/q1_3.R")

The dashed line is the nominal 5% error rate. The bars will vary from run to run.


In [ ]:
mat_q13 <- rbind(full=rates_q13$rate_full,select_then_infer=rates_q13$rate_selected)
colnames(mat_q13) <- sprintf("%s\n(r=%.2f)",rates_q13$pair,rates_q13$cor_x1_x2)
local({
    old_par <- par(mar=c(5,4.5,3,1))
    on.exit(par(old_par))
    barplot(mat_q13,beside=TRUE,col=c("steelblue","firebrick"),
            ylim=c(0,max(.15,max(mat_q13)*1.3)),
            ylab="Estimated Type I error rate",main="Fixed model and select-then-infer")
    abline(h=.05,lty=2)
    legend("topleft",legend=c("Full model","Select-then-infer"),
           fill=c("steelblue","firebrick"),bty="n")
})


**Question 1.4.** Which statement best describes the long-run pattern in these simulations? Set `q1_4_ans` to one number.

1. The fixed full-model test stays near 5%. Selection can inflate the reported test's false-positive rate, with more distortion for these more correlated pairs.
2. Both tests must have inflated rates because `Hitters` is an observational dataset.
3. Dropping a predictor with a large p-value guarantees a correctly calibrated test of the remaining predictor.


_Points:_ 1

In [ ]:
q1_4_ans <- NULL # YOUR CODE HERE


In [ ]:
. = ottr::check("tests/q1_4.R")

**Question 1.5.** A selection procedure rejects 8% of the time when its null is true, even though it uses a 5% cutoff. Its inflation factor is $0.08/0.05=1.6$. What does this mean? Set `q1_5_ans` to one number.

1. The procedure rejects the null in 60% of datasets.
2. False positives occur about 60% more often than the nominal rate: about eight rather than five per 100 null datasets.
3. A rejected null has a 60% chance of being false.
4. The estimated slope is biased upward by 60%.


_Points:_ 1

In [ ]:
q1_5_ans <- NULL # YOUR CODE HERE


In [ ]:
. = ottr::check("tests/q1_5.R")

## Part 2: Primary and Secondary Analyses

Is salary associated with years in the major leagues after accounting for 1986 performance, league, and division? For this exercise, assume we chose the following model in advance:
$$
\texttt{Salary} \sim \texttt{Years} + \texttt{Hits} + \texttt{HmRun} + \texttt{Runs} + \texttt{RBI} + \texttt{Walks} + \texttt{PutOuts} + \texttt{League} + \texttt{Division}.
$$
The primary test is $H_0:\beta_{\texttt{Years}}=0$. Choosing the model in advance addresses the selection problem from Part 1; the test still relies on the regression assumptions.


**Question 2.1.** Write `fit_primary(data, alpha = 0.05)` to fit the model above. Return a list with:

* `model`: the fitted `lm` object;
* `p_value`: the two-sided `Years` p-value;
* `reject`: a logical indicating whether `p_value < alpha`.

Assume `data` has the same columns as `hitters`, with both levels of `League` and `Division` present. Use the supplied data and threshold. The model should support predictions on new data with these columns.


_Points:_ 1.5

In [ ]:
fit_primary <- function(data, alpha = 0.05) {
    # YOUR CODE HERE
}


In [ ]:
primary_result <- fit_primary(hitters)
primary_fit <- primary_result$model
pval_primary <- primary_result$p_value
reject_primary <- primary_result$reject
data.frame(p_value = pval_primary, reject = reject_primary)


In [ ]:
. = ottr::check("tests/q2_1.R")

### Changing the Adjustment Variables

Lecture 13 uses secondary analyses to check how conclusions depend on analysis choices. Here, use backward elimination to see what happens when adjustment variables are removed, keeping `Years` in the model throughout.


**Question 2.2.** Write `backward_secondary(data, alpha = 0.05)`, starting with the full primary model. At each step, remove the non-`Years` covariate with the largest coefficient p-value if that p-value is greater than `alpha`, then refit. Stop when none of the remaining adjustment variables has a p-value above `alpha`. Keep `Years` even if its p-value is large. Each factor has two levels, so use its single dummy coefficient's p-value.

Return a list with `model`, the final `lm` fit; `p_value`, its `Years` p-value; and `variables`, the retained covariate names in any order. Use column names such as `Division` for factors and exclude the intercept. Your function should also work with other input data and thresholds.


_Points:_ 2.5

In [ ]:
backward_secondary <- function(data, alpha = 0.05) {
    # YOUR CODE HERE
}


In [ ]:
secondary_result <- backward_secondary(hitters)
final_fit <- secondary_result$model
pval_sensitivity <- secondary_result$p_value
final_model_vars <- secondary_result$variables
data.frame(analysis = c("Primary", "Secondary: backward elimination"),
           years_p_value = c(pval_primary, pval_sensitivity))
final_model_vars


In [ ]:
. = ottr::check("tests/q2_2.R")

**Question 2.3.** We used backward elimination to choose `final_fit`. What does its usual `Years` p-value fail to account for? Set `q2_3_ans` to one number.

1. R uses the wrong residual degrees of freedom after a covariate is dropped.
2. The same observations were used to choose the model and test `Years`; the usual test assumes the model was fixed in advance.
3. The remaining coefficients lose their interpretation when a covariate is removed.
4. Removing a covariate always reduces the `Years` standard error.


_Points:_ 1

In [ ]:
q2_3_ans <- NULL # YOUR CODE HERE


In [ ]:
. = ottr::check("tests/q2_3.R")

**Question 2.4.** Suppose backward elimination gives a smaller `Years` p-value than the primary model. How should the two analyses be reported? Set `q2_4_ans` to one number.

1. Replace the primary result with the selected model because its p-value is smaller.
2. Treat the two p-values as independent evidence for the same conclusion.
3. Report the primary result and present the model comparison as a secondary sensitivity analysis.
4. Report that every removed covariate has a population coefficient of zero.


_Points:_ 1

In [ ]:
q2_4_ans <- NULL # YOUR CODE HERE


In [ ]:
. = ottr::check("tests/q2_4.R")

We can report the selected model as a secondary analysis. Its smaller p-value does not change which analysis was primary.


## Part 3: Transformations

A log transformation can help when a straight line fits poorly or the residual spread changes with the mean. Following Lecture 14, compare models for `Salary` and `log(Salary)`. The histograms below show how logging changes the response distribution; the residual plots later in this part will help us assess the regression fits.


In [ ]:
par(mfrow = c(1, 2))
hist(hitters$Salary, breaks = 30, col = "steelblue",
     xlab = "Salary (thousands $)", main = "Salary")
hist(log(hitters$Salary), breaks = 30, col = "darkorange",
     xlab = "log(Salary)", main = "log(Salary)")
par(mfrow = c(1, 1))

**Question 3.1.** Write `fit_salary_models(data)` to fit `Salary ~ Years` and `log(Salary) ~ Years` to the input data. Return a list with:

* `untransformed` and `logged`: the two fitted `lm` models;
* `pval_untransformed` and `pval_logged`: their two-sided `Years` p-values.

The input has positive salaries and at least two different `Years` values. Both fits should work with `predict()` at new `Years` values.


_Points:_ 1.5

In [ ]:
fit_salary_models <- function(data) {
    # YOUR CODE HERE
}


In [ ]:
salary_models <- fit_salary_models(hitters)
fit_untransformed <- salary_models$untransformed
fit_logged <- salary_models$logged
pval_untransformed <- salary_models$pval_untransformed
pval_logged <- salary_models$pval_logged
data.frame(model=c("Salary", "log(Salary)"), p_value=c(pval_untransformed,pval_logged))


In [ ]:
. = ottr::check("tests/q3_1.R")

The residual plots below can reveal curvature or changing spread left after fitting each line.


In [ ]:
local({
    old_par <- par(mfrow=c(1,2),mar=c(4.5,4.5,2.5,1))
    on.exit(par(old_par))
    plot(fitted(fit_untransformed),residuals(fit_untransformed),pch=19,col=adjustcolor("steelblue",.6),
         xlab="Fitted salary (thousands of dollars)",ylab="Residual",main="Salary model")
    abline(h=0,lty=2)
    plot(fitted(fit_logged),residuals(fit_logged),pch=19,col=adjustcolor("darkorange",.6),
         xlab="Fitted log(salary)",ylab="Residual",main="Log-salary model")
    abline(h=0,lty=2)
})


### Simulating salaries

For the simulation, generate salaries from
$$\log(\mathrm{salary})=\beta_0+\beta_1\mathrm{years}+\varepsilon,
\qquad \varepsilon\sim N(0,\sigma^2).$$
Draw `years` independently from Uniform(1, 24), and use independent Normal errors with SD `sigma`. The relationship is linear on the log scale; the raw-scale mean is curved when the slope is nonzero.

The next cell takes the coefficients and residual SD from `fit_logged` as parameter values for this artificial population.


In [ ]:
beta0_truth <- coef(fit_logged)[1]
beta1_truth <- coef(fit_logged)[2]
sigma_truth <- summary(fit_logged)$sigma

cat("beta0 (intercept on log scale)  =", round(beta0_truth, 3), "\n")
cat("beta1 (slope on log scale)      =", round(beta1_truth, 3), "\n")
cat("sigma (residual SD on log scale)=", round(sigma_truth, 3), "\n")

**Question 3.2.** Write `sim_power_logy(reps, n, beta1, sigma, alpha=0.05)` to estimate the two tests' rejection rates under this model. Simulate `reps` datasets of size `n`, using the supplied `beta1` and `sigma` and the intercept `beta0_truth`. Draw new predictor values and errors in each replicate.

On each dataset, apply the two-sided slope tests from `salary ~ years` and `log_salary ~ years` at cutoff `alpha`. Return a list with entries `power_untransformed` and `power_logged` containing the fractions that reject.

These fractions estimate power when the slope is nonzero and Type I error when it is zero.


_Points:_ 3.5

In [ ]:
sim_power_logy <- function(reps, n, beta1, sigma, alpha=0.05) {
    # YOUR CODE HERE
}


In [ ]:
. = ottr::check("tests/q3_2.R")

In [ ]:
set.seed(414)
pow_q33 <- sim_power_logy(reps=1000,n=50,beta1=beta1_truth/4,sigma=sigma_truth)
power_q33 <- data.frame(power_untransformed=pow_q33$power_untransformed,power_logged=pow_q33$power_logged)
power_q33


**Question 3.3.** Why use repeated simulations to compare power instead of comparing the two p-values from `Hitters`? Set `q3_3_ans` to one number.

1. A test's power is one minus its observed p-value.
2. Power is the chance of rejection under a specified alternative. One observed pair of p-values doesn't tell us that chance.
3. Using the same players makes the two observed p-values equal.
4. A simulation verifies that its model describes the real players.


_Points:_ 0.5

In [ ]:
q3_3_ans <- NULL # YOUR CODE HERE


In [ ]:
. = ottr::check("tests/q3_3.R")

**Question 3.4.** Let $\widehat\beta_1$ be the `Years` slope in `lm(log(Salary) ~ Years)`. How does one more year of experience change the exponentiated prediction? Set `q3_4_ans` to one number.

1. It adds $\widehat\beta_1$ thousand dollars.
2. It multiplies the value by $\exp(\widehat\beta_1)$, a percentage change of $100\{\exp(\widehat\beta_1)-1\}\%$.
3. It adds $\log(\widehat\beta_1)$ thousand dollars.
4. It multiplies the value by $\widehat\beta_1$.


_Points:_ 1

In [ ]:
q3_4_ans <- NULL # YOUR CODE HERE


In [ ]:
. = ottr::check("tests/q3_4.R")

**Question 3.5.** Suppose the logged-model test rejects more often in this simulation. Which explanation fits the model you simulated? Set `q3_5_ans` to one number.

1. A right-skewed response always invalidates a slope test on the raw scale.
2. We simulated a linear relationship with independent Normal errors of constant variance on the log scale, so the logged test's assumptions hold.
3. Taking logs always lowers the slope p-value.
4. Whichever model gives the smaller p-value should become the primary analysis.


_Points:_ 0.5

In [ ]:
q3_5_ans <- NULL # YOUR CODE HERE


In [ ]:
. = ottr::check("tests/q3_5.R")

The rejection rates will change from run to run, and a small gap can reverse. This comparison applies to the population we simulated. Choosing a response scale simply to get a smaller p-value brings us back to the selection problem from Part 1.


---

## Submit your notebook

1. Restart the kernel and run all cells. In Jupyter, use `Kernel -> Restart Kernel and Run All Cells...` or the equivalent command in your interface.
2. Check for errors and make sure every grader check passes. Fix the first error before rerunning from the top.
3. Save the notebook, download the `.ipynb` file using `File -> Download` (or its equivalent), and upload it to Gradescope.
4. Wait for grading to finish and check the results on Gradescope, too. Passing the notebook checks does not confirm that the upload was graded successfully.
5. Confirm that Gradescope shows your submission and keep the confirmation email, if one is provided.
